# 05 - Figure 5: state-structured survival

**Question.** Does age-dependent switching among low-, mid-, and
high-order states reproduce the state composition of surviving runs?

| Panel | Analysis |
|---|---|
| A | Age-banded order-state transition probabilities |
| B | Survivor-conditioned current-state composition |
| C | Cumulative state exposure among surviving runs |

Cache mode derives every panel from the cross-fitted model outputs;
reviewer mode uses the tracked panel source tables.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# Locate the repository before importing its analysis package. This works when
# Jupyter starts from either the repo root or this notebook directory.
_start = Path.cwd()
ROOT = next(
    path for path in (_start, *_start.parents)
    if (path / "analysis" / "levy_paper").is_dir()
    and (path / "requirements.txt").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.levy_paper.util.publication_notebook_utils import (
    PRIMARY_CACHE_SUFFIX,
    cache_path as make_cache_path,
    csv_shapes,
    display_live_or_frozen,
    file_status,
    hazard_support_summary,
    load_processed_cache,
    order_state_summary,
    panel_inventory,
    publication_paths,
    relative_path,
    resolve_data_mode,
    table_inventory,
    transition_row_sum_audit,
    transport_run_summary,
)

PATHS = publication_paths(ROOT)
LEVY_DIR = PATHS["levy_dir"]
DATA_DIR = PATHS["data_dir"]
PRIMARY_CACHE_DIR = PATHS["primary_cache_dir"]
FINAL_FIGURES = PATHS["final_figures"]
SOURCE_DATA = PATHS["source_data"]
SUPPLEMENT = PATHS["supplement"]
CACHE_SUFFIX = PRIMARY_CACHE_SUFFIX

# DATA_MODE options:
#   "auto"     use processed caches when all required files exist;
#              otherwise use tracked reviewer tables/frozen figures
#   "cache"    require processed caches and fail clearly if they are missing
#   "reviewer" use only tracked public artefacts
DATA_MODE = "auto"
BUILD_FIGURE = True
SAVE_FIGURE_OUTPUTS = True
DISPLAY_FROZEN_OUTPUT = True
REBUILD_CACHE_FROM_AWS = False
REFIT_FIGURE4_FIGURE5_MODELS = False
USE_VERSIONED_FINAL_FIGURE4_FIT = True


def rel(path):
    return relative_path(path, ROOT)


def show_file_status(paths):
    return file_status(paths, ROOT)


def show_csv_shapes(paths):
    return csv_shapes(paths, ROOT)


def cache_path(stem):
    return make_cache_path(PRIMARY_CACHE_DIR, stem, CACHE_SUFFIX)


def show_figure(fig, frozen_path, width=1100):
    return display_live_or_frozen(
        fig,
        frozen_path,
        display_frozen=DISPLAY_FROZEN_OUTPUT,
        width=width,
    )

## 1. Select the model-cache pathway

In [ ]:
model_cache = DATA_DIR / "processed" / "figure4_fig5_crossfitted" / "out_of_fold_interval_predictions.parquet"
if REFIT_FIGURE4_FIGURE5_MODELS:
    if DATA_MODE == "reviewer":
        raise ValueError("Model refitting is incompatible with reviewer mode.")
    if not cache_path("hazard_intervals").exists():
        raise FileNotFoundError("The hazard-interval cache is required for refitting.")
    subprocess.run([sys.executable, str(LEVY_DIR / "scripts" / "build_figure4_figure5_model_cache.py")], cwd=ROOT, check=True)
else:
    print("model refit skipped", rel(model_cache), model_cache.exists())

resolved_mode = resolve_data_mode(DATA_MODE, [model_cache])
print("resolved_data_mode", resolved_mode)
display(show_file_status([cache_path("hazard_intervals"), model_cache]))

## 2. Derive panel and transition tables

In [ ]:
from analysis.levy_paper.scripts import create_final_fig4_fig5_polished as figure45

if resolved_mode == "cache":
    panel_bundle = figure45.build_main_panel_sources()
    panel_a = panel_bundle["figure5a"]
    panel_b = panel_bundle["figure5b"]
    panel_c = panel_bundle["figure5c"]
    transitions = panel_bundle["tables"]["transition_matrices"]
    validation = panel_bundle["tables"]["validation_metrics"]
    panel_source_mode = "derived_from_processed_crossfit_model_cache"
else:
    panel_a = pd.read_csv(SOURCE_DATA / "figure5A_source_data.csv")
    panel_b = pd.read_csv(SOURCE_DATA / "figure5B_source_data.csv")
    panel_c = pd.read_csv(SOURCE_DATA / "figure5C_source_data.csv")
    transitions = pd.read_csv(SOURCE_DATA / "crossfit_transition_parameters.csv")
    validation = pd.read_csv(SOURCE_DATA / "crossfit_validation_metrics.csv")
    panel_source_mode = "tracked_publication_source_table_fallback"

print("panel_source_mode", panel_source_mode)
display(panel_inventory({
    "Panel A": panel_a,
    "Panel B": panel_b,
    "Panel C": panel_c,
    "transition parameters": transitions,
}))

## 3. Panel A - age-dependent transition probabilities

In [ ]:
display(transition_row_sum_audit(transitions))
transition_columns = [
    column for column in [
        "age_band", "current_state", "next_state", "point_estimate",
        "lower_uncertainty_bound", "upper_uncertainty_bound",
        "n_origin_state_transitions",
    ] if column in transitions
]
display(transitions[transition_columns].head(18))

## 4. Panel B - survivor-conditioned current state

In [ ]:
panel_b_summary = panel_b.groupby("model_name", observed=True).agg(
    points=("age_s", "size"),
    min_age_s=("age_s", "min"),
    max_age_s=("age_s", "max"),
).reset_index()
display(panel_b_summary)
display(panel_b.head(12))

## 5. Panel C - survivor state exposure

In [ ]:
panel_c_summary = panel_c.groupby("model_name", observed=True).agg(
    points=("threshold_s", "size"),
    min_threshold_s=("threshold_s", "min"),
    max_threshold_s=("threshold_s", "max"),
).reset_index()
display(panel_c_summary)
display(panel_c.head(12))

## 6. Held-out validation

In [ ]:
display(validation)

## 7. Construct the publication figure

In [ ]:
fig = None
if BUILD_FIGURE:
    figure45.configure_style()
    created = []
    fig = figure45.plot_figure5(
        panel_a, panel_b, panel_c, created,
        close_figure=False,
        save_outputs=SAVE_FIGURE_OUTPUTS,
    )
display_mode = show_figure(fig, FINAL_FIGURES / "figure5_state_structured_survival.png")
print("figure_display_mode", display_mode)